# Notebook 1 — Synthetic Data & EDA

Generates 2 years of weekly marketing data across 9 channels and explores it before modelling.

Channels: **TV, Search, Social, Display, Video, OOH, Radio, Email, Affiliate**

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

from src.transforms import adstock, hill

np.random.seed(42)
sns.set_theme(style='whitegrid')
%matplotlib inline

os.makedirs('../data', exist_ok=True)
os.makedirs('../outputs', exist_ok=True)

## 1. Generate synthetic weekly data

In [ ]:
N_WEEKS = 104  # 2 years of weekly data
dates = pd.date_range('2023-01-02', periods=N_WEEKS, freq='W-MON')

t = np.arange(N_WEEKS)

# Trend + seasonality
trend    = 1.0 + 0.003 * t
seasonal = (1.0
            + 0.15 * np.sin(2 * np.pi * t / 52)
            + 0.08 * np.sin(4 * np.pi * t / 52)
            + 0.04 * np.cos(2 * np.pi * t / 52))

# Holiday uplifts: Christmas run-up (both years) and Easter (both years)
holiday = np.zeros(N_WEEKS)
for w in [49, 50, 51, 52, 101, 102, 103]:
    if w < N_WEEKS:
        holiday[w] = 0.30
for w in [12, 13, 14, 64, 65, 66]:
    if w < N_WEEKS:
        holiday[w] = 0.12

base_revenue = 500  # £000s per week at baseline
baseline = base_revenue * trend * (seasonal + holiday)

# --- Channel spend (£000s / week) ---
# TV: flighted bursts (~65% of weeks active)
tv_flight     = np.random.binomial(1, 0.65, N_WEEKS)
tv_spend      = np.random.gamma(shape=4,  scale=70, size=N_WEEKS) * tv_flight

search_spend  = np.random.gamma(shape=10, scale=13, size=N_WEEKS)  # always-on performance
social_spend  = np.random.gamma(shape=6,  scale=18, size=N_WEEKS)
display_spend = np.random.gamma(shape=4,  scale=8,  size=N_WEEKS)
video_spend   = np.random.gamma(shape=3,  scale=22, size=N_WEEKS)

# OOH: monthly campaign bursts (~50% of weeks active)
ooh_flight    = np.random.binomial(1, 0.50, N_WEEKS)
ooh_spend     = np.random.gamma(shape=3,  scale=28, size=N_WEEKS) * ooh_flight

radio_spend      = np.random.gamma(shape=4,  scale=14, size=N_WEEKS)
email_spend      = np.random.gamma(shape=12, scale=2,  size=N_WEEKS)  # owned, low cost
affiliate_spend  = np.random.gamma(shape=6,  scale=9,  size=N_WEEKS)

# Ground-truth channel parameters used to generate revenue
CHANNEL_PARAMS = {
    'tv':        dict(coef=4.0, decay=0.65, ec=200, slope=1.5),  # brand, long carry-over
    'search':    dict(coef=6.0, decay=0.10, ec=70,  slope=2.2),  # intent, fast decay
    'social':    dict(coef=3.0, decay=0.35, ec=55,  slope=1.8),
    'display':   dict(coef=1.5, decay=0.20, ec=25,  slope=1.6),  # cheap, low value
    'video':     dict(coef=2.5, decay=0.45, ec=80,  slope=1.7),
    'ooh':       dict(coef=2.0, decay=0.55, ec=120, slope=1.4),  # brand, long carry-over
    'radio':     dict(coef=1.8, decay=0.38, ec=60,  slope=1.6),
    'email':     dict(coef=4.5, decay=0.05, ec=15,  slope=2.5),  # owned, immediate
    'affiliate': dict(coef=3.5, decay=0.08, ec=30,  slope=2.0),  # performance, fast decay
}

spend_cols = [f'{ch}_spend' for ch in CHANNEL_PARAMS]
colors     = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12',
              '#9b59b6', '#1abc9c', '#e67e22', '#e91e63', '#607d8b']

spends = [tv_spend, search_spend, social_spend, display_spend,
          video_spend, ooh_spend, radio_spend, email_spend, affiliate_spend]

channel_contrib = sum(
    p['coef'] * hill(adstock(s, decay=p['decay']), ec=p['ec'], slope=p['slope'])
    for s, p in zip(spends, CHANNEL_PARAMS.values())
)

revenue = baseline + channel_contrib + np.random.normal(0, 25, N_WEEKS)

df = pd.DataFrame({'date': dates, 'revenue': revenue,
                   **{col: s for col, s in zip(spend_cols, spends)}})

df.to_csv('../data/synthetic_mmm.csv', index=False)
print(df.shape)
df.head()

## 2. Revenue over time

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(df['date'], df['revenue'], color='steelblue', linewidth=1.5)
ax.set_title('Weekly Revenue', fontsize=14)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'£{x:,.0f}k'))
plt.tight_layout()
plt.savefig('../outputs/revenue_timeseries.png', dpi=150)
plt.show()

## 3. Media spend by channel

In [ ]:
n_cols = 3
n_rows = -(-len(spend_cols) // n_cols)  # ceiling division

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 3), sharex=True)
axes_flat = list(axes.flat)

for ax, col, color in zip(axes_flat, spend_cols, colors):
    ax.fill_between(df['date'], df[col], alpha=0.6, color=color)
    ax.plot(df['date'], df[col], color=color, linewidth=0.8)
    ax.set_ylabel(col.replace('_spend', '').title(), fontsize=10)
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'£{x:,.0f}k'))

for ax in axes_flat[len(spend_cols):]:
    ax.set_visible(False)

fig.suptitle('Weekly Media Spend by Channel', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../outputs/spend_by_channel.png', dpi=150)
plt.show()

## 4. Spend share (pie)

In [ ]:
totals = df[spend_cols].sum()
labels = [c.replace('_spend','').title() for c in spend_cols]

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(totals, labels=labels, autopct='%1.1f%%', colors=colors,
       startangle=140, wedgeprops=dict(edgecolor='white', linewidth=1.5))
ax.set_title('Total Spend Share by Channel', fontsize=13)
plt.tight_layout()
plt.savefig('../outputs/spend_share_pie.png', dpi=150)
plt.show()

## 5. Correlation heatmap

In [ ]:
corr = df[['revenue'] + spend_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, ax=ax)
ax.set_title('Correlation Matrix', fontsize=13)
plt.tight_layout()
plt.savefig('../outputs/correlation_heatmap.png', dpi=150)
plt.show()

## 6. Summary stats

In [ ]:
from IPython.display import display

summary = df[['revenue'] + spend_cols].describe().T
summary['total'] = df[['revenue'] + spend_cols].sum()
display(summary.style.format('{:,.1f}'))